In [39]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel
from typing import Literal

load_dotenv()

llm_openai = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
str_parser = StrOutputParser()

class llm_schema(BaseModel):
  movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

In [40]:
# Task 1 - Prompt

prompt_template = ChatPromptTemplate.from_messages([
("system", "You are a movie review evalvator"),
("user", "Please categorize the movie review as positive or negative : {input}")
])

# Task 2 - LLM (Structured Output)


# Task 3 - Custom Runnable 
def pydantic_json(input : llm_schema) -> dict: 
  # Return a dictionary with the key "text"
  return {"text": input.model_dump()['movie_summary_flag']}

pydantic_json_lambda = RunnableLambda(pydantic_json)

# **Conditional Chain -1**

In [41]:
def linkedin_chain(text:dict):
  text = text["text"]

  # Task 1 - Prompt 
  linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are linkedin prompt generator"),
    ("user", "Create a post for the following text for Linkedin : {text}")
  ])

  # Task 2 - LLM

  # Task 3 - Str Parser

  chain_linkedin = linkedin_prompt | llm_openai | str_parser

  result = chain_linkedin.invoke(text)
  return result

chain_linkedin_runnable = RunnableLambda(linkedin_chain)

# **Conditional Chain - 2**

In [42]:
def insta_chain(text:dict):
  text = text["text"]

  # Task 1 - Prompt
  insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an instagram post generator"),
    ("user", "Create a post for the following text for Instagram: {text}")
  ])

  # Task 2 - LLM
  
  # Task 3 - Str Parser

  chain_insta = insta_prompt | llm_openai | str_parser

  result = chain_insta.invoke(text)
  return result

chain_insta_runnable = RunnableLambda(insta_chain)




# **Final Orchestration**

In [45]:
conditional_chain = RunnableBranch(
  (lambda x: "positive" in x["text"], chain_linkedin_runnable), 
  chain_insta_runnable
  )

final_chain = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

final_chain.invoke({"input": "I love aryton senna"})

'Absolutely! Here’s a positive LinkedIn post based on your request:\n\n---\n\n🌟 Embracing Positivity in Every Step 🌟\n\nIn today’s fast-paced world, maintaining a positive mindset is more important than ever. Positivity fuels creativity, drives resilience, and opens doors to new opportunities. Let’s choose to see challenges as growth moments and celebrate every small win along the way.\n\nRemember, a positive attitude not only transforms your work but also inspires those around you. Together, we can build a supportive and uplifting professional community.\n\nKeep shining and stay positive! ✨\n\n#Positivity #GrowthMindset #Leadership #Inspiration #ProfessionalDevelopment\n\n---\n\nWould you like me to tailor it to a specific industry or role?'